In [1]:
import torch
!pip install ultralytics==8.4.117 
import ultralytics

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Ultralytics version:", ultralytics.__version__)
print("Loaded from:", ultralytics.__file__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.8 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Ultralytics version: 8.4.117
Loaded from: /usr/local/lib/python3.12/dist-packages/ultralytics/__init__.py


In [2]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

candidates = []

for images_dir in INPUT_ROOT.rglob("train/images"):

    root = images_dir.parent.parent

    if (
        (root / "train/labels").exists()
        and
        (root / "val/images").exists()
        and
        (root / "val/labels").exists()
    ):
        candidates.append(root)


print("Dataset candidates:")

for i, root in enumerate(candidates):
    print(i, root)


if len(candidates) != 1:
    raise RuntimeError(
        f"Expected 1 dataset, found {len(candidates)}"
    )

DATASET_ROOT = candidates[0]

print("\nUsing:")
print(DATASET_ROOT)

Dataset candidates:
0 /kaggle/input/datasets/justinalmadrones/finalna/finalnapls

Using:
/kaggle/input/datasets/justinalmadrones/finalna/finalnapls


In [3]:
print("Train images:",
      (DATASET_ROOT / "train/images").exists())

print("Train labels:",
      (DATASET_ROOT / "train/labels").exists())

print("Val images:",
      (DATASET_ROOT / "val/images").exists())

print("Val labels:",
      (DATASET_ROOT / "val/labels").exists())

Train images: True
Train labels: True
Val images: True
Val labels: True


In [4]:
import yaml
from pathlib import Path

DATA_YAML = Path(
    "/kaggle/working/pothole_data.yaml"
)

data_config = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "names": {
        0: "pothole"
    }
}

if (DATASET_ROOT / "test/images").exists():
    data_config["test"] = "test/images"

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(
        data_config,
        f,
        sort_keys=False
    )

print(DATA_YAML.read_text())

path: /kaggle/input/datasets/justinalmadrones/finalna/finalnapls
train: train/images
val: val/images
names:
  0: pothole



In [5]:
from ultralytics import YOLO

model = YOLO(
    "yolov8s-seg.yaml",
    task="segment"
)

model.info(verbose=True)

YOLOv8s-seg summary: 152 layers, 11,821,056 parameters, 11,821,040 gradients, 40.3 GFLOPs


(152, 11821056, 11821040, 40.3390976)

In [6]:
from ultralytics import YOLO

model = YOLO(
    "yolov8s-seg.yaml",
    task="segment"
)

results = model.train(

    data=str(DATA_YAML),

    epochs=300,
    patience=30,

    imgsz=640,
    batch=4,

    device=0,
    workers=2,

    optimizer="AdamW",

    lr0=0.001,
    lrf=0.01,
    cos_lr=True,

    momentum=0.937,
    weight_decay=0.0005,

    warmup_epochs=3,

    seed=42,
    deterministic=True,

    amp=True,

    mask_ratio=4,
    overlap_mask=True,

    val=True,
    plots=True,

    save=True,
    save_period=10,

    project="/kaggle/working/runs/segment",

    name="yolov8s-baseline-300",

    exist_ok=False
)

New https://pypi.org/project/ultralytics/8.4.123 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/pothole_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=

In [7]:
from pathlib import Path

RUN_DIR = Path(
    "/kaggle/working/runs/segment/"
    "yolov8s-baseline-300"
)

print("Run exists:", RUN_DIR.exists())
print("Run directory:", RUN_DIR)

Run exists: True
Run directory: /kaggle/working/runs/segment/yolov8s-baseline-300


In [8]:
important_files = [
    RUN_DIR / "weights" / "best.pt",
    RUN_DIR / "weights" / "last.pt",
    RUN_DIR / "results.csv",
    RUN_DIR / "args.yaml",
]

print("=" * 60)
print("BASELINE FILE CHECK")
print("=" * 60)

for file in important_files:
    print(
        "✅" if file.exists() else "❌",
        file.name
    )

BASELINE FILE CHECK
✅ best.pt
✅ last.pt
✅ results.csv
✅ args.yaml


In [9]:
import shutil
from pathlib import Path

BASELINE_ZIP = Path(
    "/kaggle/working/"
    "yolov8s-baseline-300-results"
)

shutil.make_archive(
    str(BASELINE_ZIP),
    "zip",
    str(RUN_DIR)
)

print("✅ Baseline ZIP created:")
print(str(BASELINE_ZIP) + ".zip")

✅ Baseline ZIP created:
/kaggle/working/yolov8s-baseline-300-results.zip


In [15]:
from IPython.display import FileLink

FileLink(r"yolov8s-baseline-300-results.zip")

/kaggle/working/yolov8s-baseline-300-results.zip

In [16]:
import pandas as pd
from pathlib import Path

RUN_DIR = Path(
    "/kaggle/working/runs/segment/yolov8s-baseline-300"
)

CSV_PATH = RUN_DIR / "results.csv"

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

# Find epoch with highest Mask mAP50-95
best_idx = df["metrics/mAP50-95(M)"].idxmax()
best = df.loc[best_idx]

print("=" * 60)
print("BEST BASELINE YOLOv8s-SEG RESULT")
print("=" * 60)

print("Epoch:", int(best["epoch"]))

print("\nMASK RESULTS")
print(f'Precision:   {best["metrics/precision(M)"] * 100:.2f}%')
print(f'Recall:      {best["metrics/recall(M)"] * 100:.2f}%')
print(f'mAP50:       {best["metrics/mAP50(M)"] * 100:.2f}%')
print(f'mAP50-95:    {best["metrics/mAP50-95(M)"] * 100:.2f}%')

print("\nBOX RESULTS")
print(f'Precision:   {best["metrics/precision(B)"] * 100:.2f}%')
print(f'Recall:      {best["metrics/recall(B)"] * 100:.2f}%')
print(f'mAP50:       {best["metrics/mAP50(B)"] * 100:.2f}%')
print(f'mAP50-95:    {best["metrics/mAP50-95(B)"] * 100:.2f}%')

BEST BASELINE YOLOv8s-SEG RESULT
Epoch: 145

MASK RESULTS
Precision:   84.97%
Recall:      78.18%
mAP50:       82.86%
mAP50-95:    53.91%

BOX RESULTS
Precision:   83.78%
Recall:      77.46%
mAP50:       82.09%
mAP50-95:    56.29%
